# 03 · STGCN on METR-LA

Spatio-Temporal Graph Convolutional Network (Yu, Yin & Zhu, IJCAI 2018), implemented in `src/models/stgcn/`.

Runs on **Google Colab** (use a GPU runtime) or **locally** (CUDA, Apple MPS or CPU).
Requires the processed files from `01_data_preprocess.ipynb`.

The notebook builds the graph and model, then trains with validation-based early stopping
(or reuses a finished run in `results/`), and evaluates on the held-out test period with
MAE, RMSE and MAPE at 15, 30 and 60 minutes. The same training code runs from the command line:
`python -m src.models.stgcn.train --adjacency distance`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mansiguptacs/CMPE_258_TrafficForecastingUsing_GraphNN.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/CMPE_258_TrafficForecastingUsing_GraphNN")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tables", "pyyaml"], check=True)
else:
    repo = Path.cwd()
    while not (repo / "configs" / "metr_la.yaml").exists():
        if repo.parent == repo:
            raise RuntimeError("Start this notebook from inside the project repository")
        repo = repo.parent

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Colab" if IN_COLAB else "Local", "| repo:", repo)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import load_config, processed_dir
from src.data.pipeline import load_processed, make_datasets, PROCESSED_FILE
from src.eval.metrics import horizon_metrics
from src.models.stgcn.graph import get_adjacency, scaled_laplacian
from src.models.stgcn.train import build_model, train, run_dir, pick_device

cfg = load_config()
if not (processed_dir(cfg) / PROCESSED_FILE).exists():
    raise FileNotFoundError("Processed data not found. Run 01_data_preprocess.ipynb first.")

device = pick_device()
data = load_processed(cfg)
datasets = make_datasets(cfg, data)
interval = cfg["dataset"]["interval_minutes"]
print("device:", device)
print({split: len(ds) for split, ds in datasets.items()}, "windows")

## 1. Graph

Chebyshev graph convolution needs the scaled Laplacian
$\tilde L = 2L/\lambda_{max} - I$, with $L = I - D^{-1/2} W D^{-1/2}$.
The adjacency is chosen by `graph.adjacency` in the config (`distance` or `correlation`).
The road adjacency is directed, so it is symmetrized for STGCN.

In [ ]:
adj = get_adjacency(cfg, data)
lap = scaled_laplacian(adj)
eig = np.linalg.eigvalsh(lap)

sym = np.maximum(adj, adj.T)
np.fill_diagonal(sym, 0)
degree = (sym > 0).sum(axis=1)
print(f"adjacency: {cfg['graph']['adjacency']}, {int((sym > 0).sum() // 2)} undirected edges")
print(f"neighbours per sensor: mean {degree.mean():.1f}, min {degree.min()}, max {degree.max()}")
print(f"scaled Laplacian eigenvalues in [{eig.min():.2f}, {eig.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(sym > 0, cmap="Greys", interpolation="nearest")
axes[0].set(title="Sensor connectivity (symmetrized)", xlabel="sensor", ylabel="sensor")
axes[1].hist(degree, bins=20)
axes[1].set(title="Neighbours per sensor", xlabel="degree", ylabel="sensors")
plt.tight_layout(); plt.show()

## 2. Model

Two ST-Conv blocks, each: gated temporal conv → Chebyshev graph conv → gated temporal conv → layer norm.
With 12 input steps and temporal kernel 3, the blocks shrink time from 12 to 4; the output layer
collapses the remaining 4 steps and predicts all 12 future steps at once.

In [ ]:
model = build_model(cfg, data, device)
print(model)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

x, y, y_mask = next(iter(DataLoader(datasets["train"], batch_size=64)))
with torch.no_grad():
    out = model(x.to(device))
print("input :", tuple(x.shape), "[batch, time, sensors, features]")
print("output:", tuple(out.shape), "[batch, future steps, sensors]")

## 3. Training

Loss is masked MAE in mph, so missing targets do not count. Adam with the learning rate halved
when validation error stalls, early stopping on validation MAE, best checkpoint kept.
If a finished run exists for this graph it is reused; delete `results/stgcn_<graph>/` to retrain
(about 40 s per epoch on Apple MPS).

In [ ]:
tag = f"stgcn_{cfg['graph']['adjacency']}"
out_dir = run_dir(cfg, tag)
if not (out_dir / "test_predictions.npz").exists():
    train(cfg, tag=tag)

history = pd.DataFrame(json.load(open(out_dir / "history.json")))
results = json.load(open(out_dir / "metrics.json"))
print(f"{results['epochs_run']} epochs, best epoch {results['best_epoch']}, {results['train_minutes']:.1f} min on {results['device']}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(history["epoch"], history["train_mae"], label="train")
ax.plot(history["epoch"], history["val_mae"], label="validation")
ax.axvline(results["best_epoch"], color="gray", ls=":", label="best epoch")
ax.set(title="STGCN learning curves", xlabel="epoch", ylabel="MAE (mph)")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Test results

Scored on the test period (4 June to 27 June 2012) only, on observed readings only.

In [ ]:
rows = {name: {"MAE (mph)": m["mae"], "RMSE (mph)": m["rmse"], "MAPE (%)": m["mape"]}
        for name, m in results["test"].items()}
table = pd.DataFrame(rows).T.round(3)
display(table)

val_rows = {name: {"MAE (mph)": m["mae"], "RMSE (mph)": m["rmse"], "MAPE (%)": m["mape"]}
            for name, m in results["val"].items()}
print("validation split, for reference:")
display(pd.DataFrame(val_rows).T.round(3))

In [ ]:
pred = np.load(out_dir / "test_predictions.npz")["pred"]
t0, t1 = data["meta"]["split_index"]["test"]
w = cfg["windows"]
n_win = pred.shape[0]
target = np.stack([data["speeds"][t0 + i + w["input_len"]: t0 + i + w["input_len"] + w["output_len"]]
                   for i in range(n_win)])
mask = np.stack([data["observed"][t0 + i + w["input_len"]: t0 + i + w["input_len"] + w["output_len"]]
                 for i in range(n_win)])

# error as the forecast reaches further ahead
steps = np.arange(1, w["output_len"] + 1)
mae_by_step = [horizon_metrics(pred, target, mask, horizons=(s,))[f"{s * interval}min"]["mae"] for s in steps]
rmse_by_step = [horizon_metrics(pred, target, mask, horizons=(s,))[f"{s * interval}min"]["rmse"] for s in steps]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(steps * interval, mae_by_step, marker="o", label="MAE")
ax.plot(steps * interval, rmse_by_step, marker="s", label="RMSE")
ax.set(title="STGCN test error by forecast horizon", xlabel="minutes ahead", ylabel="mph")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Predicted vs actual speeds

In [ ]:
# pick the sensor with the most variable speed on a fully observed test weekday
per_day = 24 * 60 // interval
day_start = 2 * per_day          # third day of the test period (windows are indexed by forecast origin)
day = slice(day_start, day_start + per_day)
obs_share = mask[day, 0].mean(axis=0)
spread = np.where(obs_share > 0.99, target[day, 0].std(axis=0), -1)
sensor = int(np.argmax(spread))

clock = pd.to_datetime(data["timestamps"][t0 + w["input_len"]:][:n_win])
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, h in zip(axes, (3, 12)):
    actual = np.where(mask[day, h - 1, sensor], target[day, h - 1, sensor], np.nan)
    when = clock[day] + pd.Timedelta(minutes=(h - 1) * interval)
    ax.plot(when, actual, label="actual", lw=1.5)
    ax.plot(when, pred[day, h - 1, sensor], label="STGCN", lw=1.5)
    ax.set(title=f"Sensor {data['sensor_ids'][sensor]}, {h * interval}-minute forecast", ylabel="mph")
    ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

## 6. Where does the model do worse?

In [ ]:
# per-sensor MAE (all forecast steps) against the number of neighbours in the graph
err = np.abs(pred - target) * mask
sensor_mae = err.sum(axis=(0, 1)) / np.maximum(mask.sum(axis=(0, 1)), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(sensor_mae, bins=25)
axes[0].set(title="Test MAE per sensor", xlabel="MAE (mph)", ylabel="sensors")
axes[1].scatter(degree, sensor_mae, s=14)
axes[1].set(title="Test MAE vs number of graph neighbours", xlabel="neighbours", ylabel="MAE (mph)")
plt.tight_layout(); plt.show()

print(f"sensor MAE: median {np.median(sensor_mae):.2f}, worst {sensor_mae.max():.2f} mph")
print(f"correlation between neighbours and MAE: {np.corrcoef(degree, sensor_mae)[0, 1]:.2f}")
worst = np.argsort(sensor_mae)[::-1][:5]
print("worst sensors:", [(data['sensor_ids'][i], round(float(sensor_mae[i]), 2), int(degree[i])) for i in worst],
      "(id, MAE, neighbours)")

## 7. Does the graph help? Ablation without edges

The same STGCN is trained with an empty graph (`python -m src.models.stgcn.train --adjacency identity --tag stgcn_no_graph`).
Every layer, parameter count and training setting is identical; the only change is that sensors can no longer see their neighbours.
The comparison is on the same test windows.

In [ ]:
no_graph_dir = run_dir(cfg, "stgcn_no_graph")
if not (no_graph_dir / "test_predictions.npz").exists():
    raise FileNotFoundError("Run: python -m src.models.stgcn.train --adjacency identity --tag stgcn_no_graph")
res_ng = json.load(open(no_graph_dir / "metrics.json"))
pred_ng = np.load(no_graph_dir / "test_predictions.npz")["pred"]

names = ["15min", "30min", "60min", "average"]
cmp = pd.DataFrame({
    ("MAE (mph)", "graph"): [results["test"][k]["mae"] for k in names],
    ("MAE (mph)", "no graph"): [res_ng["test"][k]["mae"] for k in names],
    ("RMSE (mph)", "graph"): [results["test"][k]["rmse"] for k in names],
    ("RMSE (mph)", "no graph"): [res_ng["test"][k]["rmse"] for k in names],
    ("MAPE (%)", "graph"): [results["test"][k]["mape"] for k in names],
    ("MAPE (%)", "no graph"): [res_ng["test"][k]["mape"] for k in names],
}, index=names).round(3)
display(cmp)
print(f"epochs run: graph {results['epochs_run']} (best {results['best_epoch']}), "
      f"no graph {res_ng['epochs_run']} (best {res_ng['best_epoch']})")

In [ ]:
# error split by traffic state: congested = realised speed below 40 mph
def state_mae(p, h, congested):
    t, m = target[:, h - 1], mask[:, h - 1]
    sel = m & ((t < 40) if congested else (t >= 40))
    return np.abs(p[:, h - 1][sel] - t[sel]).mean(), sel.sum() / m.sum()

rows = []
for h in (3, 6, 12):
    for congested in (True, False):
        g, share = state_mae(pred, h, congested)
        n, _ = state_mae(pred_ng, h, congested)
        rows.append({"horizon": f"{h * interval} min", "traffic": "congested (<40 mph)" if congested else "free flow",
                     "share of readings": f"{share:.1%}", "MAE graph": round(g, 2), "MAE no graph": round(n, 2),
                     "graph improves by": f"{(n - g) / n:.1%}"})
state = pd.DataFrame(rows)
display(state)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
x_pos = np.arange(3); wd = 0.38
axes[0].bar(x_pos - wd / 2, [results["test"][k]["mae"] for k in names[:3]], wd, label="STGCN with graph")
axes[0].bar(x_pos + wd / 2, [res_ng["test"][k]["mae"] for k in names[:3]], wd, label="STGCN without graph")
axes[0].set(xticks=x_pos, xticklabels=["15 min", "30 min", "60 min"], ylabel="test MAE (mph)", title="Test MAE by horizon")
axes[0].legend()
cong = state[state["traffic"].str.startswith("congested")]
free = state[state["traffic"] == "free flow"]
axes[1].bar(x_pos - wd / 2, cong["MAE graph"], wd, label="graph")
axes[1].bar(x_pos + wd / 2, cong["MAE no graph"], wd, label="no graph")
axes[1].set(xticks=x_pos, xticklabels=["15 min", "30 min", "60 min"], ylabel="test MAE (mph)", title="Congested traffic only (<40 mph)")
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# per-sensor gain from the graph, against how many neighbours the sensor has
mae_ng = (np.abs(pred_ng - target) * mask).sum(axis=(0, 1)) / np.maximum(mask.sum(axis=(0, 1)), 1)
gain = mae_ng - sensor_mae
print(f"graph model is better on {(gain > 0).sum()} of {len(gain)} sensors; mean gain {gain.mean():.2f} mph, median {np.median(gain):.2f}")
print(f"correlation between gain and number of neighbours: {np.corrcoef(degree, gain)[0, 1]:.2f}")

## Next

- Compare against the historical-average, LSTM and DCRNN results under the same split and metrics.
- Switch `graph.adjacency` to `correlation` once the correlation graph is built, and compare.
- Repeat runs with different seeds to see how much of the graph / no-graph gap is training noise.
- Missing-data robustness with `src.data.missing.apply_missing_mask`.